# Phase 2: the fine-tune grid, on a Kaggle GPU

Runs the pre-registered grid in `docs/serving-rule.md` (runs A', B, C, D, seeds
13, 42, 1234) with `python -m src.finetune_grid`, then summarises it.

**Settings before running:**
- Accelerator: **GPU T4** (a single T4 is enough; see the next cell for T4 x2).
- Internet: **on** (the repository and the start checkpoints are downloaded).
- Use **Save Version -> Save & Run All (Commit)** so the run continues in the
  background, up to Kaggle's 12-hour limit, without the browser open.

No secrets or tokens are needed: the repository is public and every start
checkpoint is a public Hugging Face model.

## 1. One GPU only

With "GPU T4 x2" the Trainer splits each batch across both cards, so the
effective batch would be 64 instead of the pre-registered 32. Pinning one
device keeps the grid's hyperparameters what the rule says they are. This must
run before anything imports torch.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. The code, at the commit that holds the pre-registered rule

In [ ]:
COMMIT = "a0a50bccbfc7392288433d93f649fa4599a17d60"
!git init -q /kaggle/working/repo
%cd /kaggle/working/repo
!git remote add origin https://github.com/raviteja311/Sentiment-Analysis.git
!git fetch -q --depth 1 origin {COMMIT}
!git checkout -q FETCH_HEAD
!git log --oneline -1

## 3. Dependencies

Only what training needs, at the versions the project pins. Kaggle's own CUDA
build of torch is kept: the full requirements file would replace it and add
TensorFlow, which the grid never imports.

In [ ]:
!pip install -q transformers==4.57.3 datasets==4.4.1 accelerate==1.11.0 emoji==2.16.0 scikit-learn==1.7.2
import torch, transformers

print(torch.__version__, "CUDA:", torch.cuda.is_available(), "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "No GPU: set the accelerator to GPU T4"
assert torch.cuda.device_count() == 1, "More than one GPU visible; rerun cell 1 first"

## 4. Resume from an earlier session (optional)

If a previous version timed out, add its `finetune_grid.zip` output as an input
to this notebook. Finished cells are restored and skipped; only the rest train.

In [ ]:
import glob, zipfile

previous = glob.glob("/kaggle/input/**/finetune_grid.zip", recursive=True)
for archive in previous:
    with zipfile.ZipFile(archive) as z:
        z.extractall("reports/experiments")
    print("restored", archive)
!ls reports/experiments/finetune_grid 2>/dev/null || echo "starting fresh"

## 5. Smoke test

Two cells on 64 examples, written to a scratch directory that is never
summarised. Catches a broken environment in a minute instead of an hour in.

In [ ]:
!python -m src.finetune_grid run --runs B C --seeds 13 --limit 64

## 6. The grid

Twelve cells. Each one that finishes is on disk, so a timeout loses only the
cell in flight; resume with step 4.

In [ ]:
!python -m src.finetune_grid run

## 7. Summary and the serving decision

Applies the pre-registered rule against the base checkpoint's committed test
predictions. Prints "incomplete" if any run is missing a seed.

In [ ]:
!python -m src.finetune_grid summarise

## 8. Download the results

`finetune_grid.zip` appears under this version's Output. It holds each cell's
record, training curve and predictions (no weights), plus the summary. Unzip it
into `reports/experiments/` in the local repository and run
`python -m src.finetune_grid summarise` there to reproduce the table.

In [ ]:
import shutil

shutil.copy("reports/experiments/finetune_grid.json", "reports/experiments/finetune_grid/")
shutil.make_archive("/kaggle/working/finetune_grid", "zip", "reports/experiments", "finetune_grid")
!ls -lh /kaggle/working/finetune_grid.zip